# 115 — GPU XGBoost + CatBoost Ensemble (Kaggle T4)

Fully self-contained GPU notebook for Kaggle. Downloads data from HuggingFace.
No pxr module dependency.

Models:
- XGBoost with GPU histogram (tree_method=hist, device=cuda)
- CatBoost with GPU (task_type=GPU)

Features: Morgan ECFP4 (2048) + ECFP6 (2048) + RDKit descriptors (~200)

In [ ]:
# === Setup: data download + GPU check ===
import sys, os, subprocess, urllib.request, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path

# Kaggle paths
WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path('.')
DATA_DIR = WORK / 'rawdata'; DATA_DIR.mkdir(exist_ok=True)
SUBMISSIONS = WORK / 'submissions'; SUBMISSIONS.mkdir(exist_ok=True)
PROCESSED = WORK / 'processed'; PROCESSED.mkdir(exist_ok=True)

HF_BASE = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main'

def fetch(fname):
    p = DATA_DIR / fname
    if not p.exists():
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(f'{HF_BASE}/{fname}', p)
    return p

# Check GPU
try:
    import torch
    gpu_ok = torch.cuda.is_available()
    if gpu_ok: print(f'GPU: {torch.cuda.get_device_name(0)}')
    else: print('No GPU via torch')
except: gpu_ok = False; print('torch not available')

print(f'Python {sys.version[:6]}  GPU={gpu_ok}')

In [ ]:
# === Load data ===
tr_raw = pd.read_csv(fetch('pxr-challenge_TRAIN.csv'))
te_raw = pd.read_csv(fetch('pxr-challenge_TEST_BLINDED.csv'))

COL_MAP_TR = {'Molecule Name':'name','SMILES':'smiles','pEC50':'pec50',
               'pEC50_std.error (-log10(molarity))':'pec50_se',
               'Emax_estimate (log2FC vs. baseline)':'emax'}
COL_MAP_TE = {'Molecule Name':'name','SMILES':'smiles'}
tr = tr_raw.rename(columns=COL_MAP_TR)
te = te_raw.rename(columns=COL_MAP_TE)
tr = tr.dropna(subset=['pec50'])

y_tr = tr['pec50'].values.astype(np.float64)
print(f'Train: {len(tr):,}  Test: {len(te):,}  pEC50 range: [{y_tr.min():.2f}, {y_tr.max():.2f}]')

In [ ]:
# === Featurization (rdkit pre-installed on Kaggle) ===
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

def morgan_fps(smiles_list, radius=2, n_bits=2048):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(str(s))
        if mol:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            fps.append(list(fp))
        else:
            fps.append([0]*n_bits)
    return np.array(fps, dtype=np.float32)

DESC_NAMES = [d[0] for d in Descriptors.descList if 'Num' not in d[0][:3]][:200]
def rdkit_descs(smiles_list):
    rows = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(str(s))
        if mol:
            row = []
            for nm in DESC_NAMES:
                try: row.append(float(getattr(Descriptors, nm)(mol)) or 0.0)
                except: row.append(0.0)
        else:
            row = [0.0] * len(DESC_NAMES)
        rows.append(row)
    arr = np.array(rows, dtype=np.float32)
    # Impute: fill nan/inf with column median
    for c in range(arr.shape[1]):
        col = arr[:,c]
        bad = ~np.isfinite(col)
        if bad.any():
            med = np.nanmedian(col[~bad]) if (~bad).any() else 0.0
            arr[bad, c] = float(med) if np.isfinite(med) else 0.0
    return arr

def bemis_murcko(s):
    mol = Chem.MolFromSmiles(str(s))
    try: return MurckoScaffold.GetScaffoldForMol(mol) if mol else s
    except: return s

def scaffold_kfold(scaffolds, n_splits=5, seed=42):
    from collections import defaultdict
    s2i = defaultdict(list)
    for i, s in enumerate(scaffolds): s2i[s].append(i)
    buckets = sorted(s2i.values(), key=len, reverse=True)
    folds = [[] for _ in range(n_splits)]; sizes = [0]*n_splits
    for b in buckets:
        f = min(range(n_splits), key=lambda k: sizes[k])
        folds[f].extend(b); sizes[f] += len(b)
    all_idx = list(range(len(scaffolds)))
    return [(sorted(set(all_idx)-set(folds[k])), folds[k]) for k in range(n_splits)]

print('Computing Morgan ECFP4...')
fp4_tr = morgan_fps(tr['smiles'].tolist(), radius=2)
fp4_te = morgan_fps(te['smiles'].tolist(), radius=2)
print('Computing Morgan ECFP6...')
fp6_tr = morgan_fps(tr['smiles'].tolist(), radius=3)
fp6_te = morgan_fps(te['smiles'].tolist(), radius=3)
print('Computing RDKit descriptors...')
desc_tr = rdkit_descs(tr['smiles'].tolist())
desc_te = rdkit_descs(te['smiles'].tolist())

X_tr = np.hstack([fp4_tr, fp6_tr, desc_tr])
X_te = np.hstack([fp4_te, fp6_te, desc_te])
print(f'Feature matrix: {X_tr.shape}')

scaffolds = [bemis_murcko(s) for s in tr['smiles'].tolist()]
splits = scaffold_kfold(scaffolds, n_splits=5, seed=42)
print(f'Scaffold folds ready')

In [ ]:
# === XGBoost GPU model ===
try:
    import xgboost as xgb
    XGB_OK = True
    print(f'XGBoost {xgb.__version__}')
except ImportError:
    XGB_OK = False
    print('XGBoost not available')

def rae(yt, yp):
    yt, yp = np.asarray(yt, float), np.asarray(yp, float)
    return float(np.mean(np.abs(yt-yp)) / np.mean(np.abs(yt-yt.mean())))

if XGB_OK:
    oof_xgb = np.full(len(y_tr), np.nan)
    XGB_PARAMS = {
        'n_estimators': 2000,
        'max_depth': 7,
        'learning_rate': 0.03,
        'subsample': 0.8,
        'colsample_bytree': 0.6,
        'min_child_weight': 3,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'n_jobs': 4,
        'tree_method': 'hist',
        'device': 'cuda' if gpu_ok else 'cpu',
    }
    for fold, (tr_idx, va_idx) in enumerate(splits):
        m = xgb.XGBRegressor(**XGB_PARAMS)
        m.fit(X_tr[tr_idx], y_tr[tr_idx],
              eval_set=[(X_tr[va_idx], y_tr[va_idx])],
              verbose=False, early_stopping_rounds=50)
        oof_xgb[va_idx] = m.predict(X_tr[va_idx])
        r = rae(y_tr[va_idx], oof_xgb[va_idx])
        print(f'  fold {fold+1}  XGB RAE={r:.4f}  best_iter={m.best_iteration}', flush=True)

    r_xgb = rae(y_tr, oof_xgb)
    print(f'\nXGBoost OOF RAE: {r_xgb:.4f}')

    # Final model
    m_xgb_final = xgb.XGBRegressor(**XGB_PARAMS)
    m_xgb_final.fit(X_tr, y_tr, verbose=False)
    te_xgb = m_xgb_final.predict(X_te)
else:
    oof_xgb = None; te_xgb = None; r_xgb = float('inf')

In [ ]:
# === CatBoost GPU model ===
try:
    from catboost import CatBoostRegressor
    CB_OK = True
    print('CatBoost available')
except ImportError:
    print('Installing CatBoost...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'catboost', '-q'], check=False)
    try:
        from catboost import CatBoostRegressor
        CB_OK = True
    except: CB_OK = False; print('CatBoost unavailable')

if CB_OK:
    oof_cb = np.full(len(y_tr), np.nan)
    CB_PARAMS = {
        'iterations': 2000,
        'depth': 7,
        'learning_rate': 0.03,
        'l2_leaf_reg': 3,
        'random_seed': 42,
        'task_type': 'GPU' if gpu_ok else 'CPU',
        'verbose': False,
        'early_stopping_rounds': 50,
    }
    for fold, (tr_idx, va_idx) in enumerate(splits):
        m = CatBoostRegressor(**CB_PARAMS)
        m.fit(X_tr[tr_idx], y_tr[tr_idx],
              eval_set=(X_tr[va_idx], y_tr[va_idx]),
              verbose=False)
        oof_cb[va_idx] = m.predict(X_tr[va_idx])
        r = rae(y_tr[va_idx], oof_cb[va_idx])
        print(f'  fold {fold+1}  CB RAE={r:.4f}', flush=True)

    r_cb = rae(y_tr, oof_cb)
    print(f'\nCatBoost OOF RAE: {r_cb:.4f}')

    m_cb_final = CatBoostRegressor(**CB_PARAMS)
    m_cb_final.fit(X_tr, y_tr, verbose=False)
    te_cb = m_cb_final.predict(X_te)
else:
    oof_cb = None; te_cb = None; r_cb = float('inf')

In [ ]:
# === Ensemble + save ===
print('\n=== Results ===')
results = []
if oof_xgb is not None: results.append(('xgb', r_xgb, oof_xgb, te_xgb))
if oof_cb is not None: results.append(('catboost', r_cb, oof_cb, te_cb))
results.sort(key=lambda x: x[1])
for name, r, _, _ in results:
    print(f'  {name}: OOF RAE = {r:.4f}')

if len(results) >= 2:
    # Equal-weight ensemble
    oof_ens = np.mean([r[2] for r in results], axis=0)
    te_ens = np.mean([r[3] for r in results], axis=0)
    r_ens = rae(y_tr, oof_ens)
    print(f'  ensemble: OOF RAE = {r_ens:.4f}')
    oof_final = oof_ens; te_final = te_ens; r_final = r_ens
else:
    oof_final = results[0][2]; te_final = results[0][3]; r_final = results[0][1]

te_final = np.clip(te_final, y_tr.min()-0.5, y_tr.max()+0.5)

np.save(PROCESSED/'oof_xgb_gpu_ensemble.npy', oof_final)
np.save(PROCESSED/'te_oof_xgb_gpu_ensemble.npy', te_final)

if oof_xgb is not None:
    np.save(PROCESSED/'oof_xgb_gpu.npy', oof_xgb)
    np.save(PROCESSED/'te_oof_xgb_gpu.npy', te_xgb)
if oof_cb is not None:
    np.save(PROCESSED/'oof_catboost_gpu.npy', oof_cb)
    np.save(PROCESSED/'te_oof_catboost_gpu.npy', te_cb)

sub = pd.DataFrame({'Molecule Name': te['name'].values, 'pEC50': te_final})
assert len(sub) == 513 and sub['pEC50'].notna().all()
p = SUBMISSIONS/'115_xgboost_gpu_kaggle.csv'
sub.to_csv(p, index=False)
print(f'Saved {p}')
print(f'Test: min={te_final.min():.2f} med={np.median(te_final):.2f} max={te_final.max():.2f}')
print(f'\n*** nb115 GPU ensemble OOF RAE = {r_final:.4f} ***')